In [1]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# ---- data ----
df = pd.read_csv("../Data/imf-dm-export-20260910.csv")
df = df.rename(columns={df.columns[0]: "country"})
uk = df[df["country"] == "United Kingdom"].iloc[0]

# year columns are strings in a CSV — match 4-digit names, not int type
years = [c for c in df.columns if str(c).strip().isdigit() and len(str(c).strip()) == 4]
data = pd.DataFrame({"year": [int(y) for y in years],
                     "debt": [pd.to_numeric(uk[y], errors="coerce") for y in years]}).dropna()

# ---- who was in power ----
def party(y):
    if y < 1997: return "Conservative"
    if y < 2010: return "Labour"
    if y < 2015: return "Coalition"      # Con–Lib Dem, May 2010 – May 2015
    if y < 2024: return "Conservative"
    return "Labour"
data["party"] = data["year"].apply(party)

# collapse consecutive years into spans
spans, cur, start = [], data.iloc[0]["party"], data.iloc[0]["year"]
for _, r in data.iterrows():
    if r["party"] != cur:
        spans.append((cur, start, r["year"])); cur, start = r["party"], r["year"]
spans.append((cur, start, data["year"].max() + 1))
spans_df = pd.DataFrame(spans, columns=["party", "x0", "x1"])

# ---- colours ----
CON, LAB, LIB, INK = "#0087DC", "#E4003B", "#FAA61A", "#122b39"
XDOM = [1980, data["year"].max() + 1]
BAND_OP = 0.14          # opacity for all government shading (bands + stripes)

# solid bands for single-party governments
solid_df = spans_df[spans_df["party"] != "Coalition"]

# coalition -> alternating blue/orange vertical stripes
STRIPE = 0.5            # width of each stripe, in years
stripes = []
for _, s in spans_df[spans_df["party"] == "Coalition"].iterrows():
    x, i = s["x0"], 0
    while x < s["x1"]:
        stripes.append({"x0": x, "x1": min(x + STRIPE, s["x1"]),
                        "c": CON if i % 2 == 0 else LIB})
        x += STRIPE; i += 1
stripes_df = pd.DataFrame(stripes, columns=["x0", "x1", "c"])

# ---- chart ----
bg = alt.Chart(solid_df).mark_rect(opacity=BAND_OP).encode(
    x=alt.X("x0:Q", scale=alt.Scale(domain=XDOM, nice=False),
            axis=alt.Axis(format="d", grid=False), title=None),
    x2="x1:Q",
    color=alt.Color("party:N",
        scale=alt.Scale(domain=["Conservative", "Labour"], range=[CON, LAB]),
        legend=alt.Legend(orient="top", direction="horizontal", title=None)))

stripe_bg = alt.Chart(stripes_df).mark_rect(opacity=BAND_OP).encode(
    x="x0:Q", x2="x1:Q",
    color=alt.Color("c:N", scale=None, legend=None))

# label the coalition band (drop this block if you don't want it)
coal = spans_df[spans_df["party"] == "Coalition"].assign(
    mid=lambda d: (d["x0"] + d["x1"]) / 2)
coal_lbl = alt.Chart(coal).mark_text(
    baseline="top", dy=4, fontSize=10, color=INK).encode(
    x="mid:Q", y=alt.value(0), text=alt.value("Coalition"))

# grey wash over the projection years (2025+)
proj = pd.DataFrame({"x0": [2025], "x1": [data["year"].max() + 1]})
proj_bg = alt.Chart(proj).mark_rect(opacity=0.10, color="#676A86").encode(x="x0:Q", x2="x1:Q")

# line: solid actual, dashed projection (overlap at 2025 so they join)
actual = data[data["year"] <= 2025]
projline = data[data["year"] >= 2025]
line_a = alt.Chart(actual).mark_line(color=INK, strokeWidth=2.2).encode(
    x="year:Q", y=alt.Y("debt:Q", title="Gross debt, % of GDP"))
line_p = alt.Chart(projline).mark_line(color=INK, strokeWidth=2.2, strokeDash=[3, 3]).encode(
    x="year:Q", y="debt:Q")

chart = (bg + stripe_bg + proj_bg + line_a + line_p + coal_lbl).properties(
    width=640, height=340,
    title=alt.Title("Debt and the party in power",
                    subtitle="UK gross debt, % of GDP · shading shows government · dashed = IMF projection")
).resolve_scale(color="independent")

styles.save(charimport pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# ---- data ----
df = pd.read_csv("../Data/imf-dm-export-20260910.csv")
df = df.rename(columns={df.columns[0]: "country"})
uk = df[df["country"] == "United Kingdom"].iloc[0]

# year columns are strings in a CSV — match 4-digit names, not int type
years = [c for c in df.columns if str(c).strip().isdigit() and len(str(c).strip()) == 4]
data = pd.DataFrame({"year": [int(y) for y in years],
                     "debt": [pd.to_numeric(uk[y], errors="coerce") for y in years]}).dropna()

# ---- who was in power ----
def party(y):
    if y < 1997: return "Conservative"
    if y < 2010: return "Labour"
    if y < 2015: return "Coalition"      # Con–Lib Dem, May 2010 – May 2015
    if y < 2024: return "Conservative"
    return "Labour"
data["party"] = data["year"].apply(party)

# collapse consecutive years into spans
spans, cur, start = [], data.iloc[0]["party"], data.iloc[0]["year"]
for _, r in data.iterrows():
    if r["party"] != cur:
        spans.append((cur, start, r["year"])); cur, start = r["party"], r["year"]
spans.append((cur, start, data["year"].max() + 1))
spans_df = pd.DataFrame(spans, columns=["party", "x0", "x1"])

# ---- colours ----
CON, LAB, LIB, INK = "#0087DC", "#E4003B", "#FAA61A", "#122b39"
XDOM = [1980, data["year"].max() + 1]
BAND_OP = 0.14          # opacity for all government shading (bands + stripes)

# solid bands for single-party governments
solid_df = spans_df[spans_df["party"] != "Coalition"]

# coalition -> alternating blue/orange vertical stripes
STRIPE = 0.5            # width of each stripe, in years
stripes = []
for _, s in spans_df[spans_df["party"] == "Coalition"].iterrows():
    x, i = s["x0"], 0
    while x < s["x1"]:
        stripes.append({"x0": x, "x1": min(x + STRIPE, s["x1"]),
                        "c": CON if i % 2 == 0 else LIB})
        x += STRIPE; i += 1
stripes_df = pd.DataFrame(stripes, columns=["x0", "x1", "c"])

# ---- chart ----
bg = alt.Chart(solid_df).mark_rect(opacity=BAND_OP).encode(
    x=alt.X("x0:Q", scale=alt.Scale(domain=XDOM, nice=False),
            axis=alt.Axis(format="d", grid=False), title=None),
    x2="x1:Q",
    color=alt.Color("party:N",
        scale=alt.Scale(domain=["Conservative", "Labour"], range=[CON, LAB]),
        legend=alt.Legend(orient="top", direction="horizontal", title=None)))

stripe_bg = alt.Chart(stripes_df).mark_rect(opacity=BAND_OP).encode(
    x="x0:Q", x2="x1:Q",
    color=alt.Color("c:N", scale=None, legend=None))

# label the coalition band (drop this block if you don't want it)
coal = spans_df[spans_df["party"] == "Coalition"].assign(
    mid=lambda d: (d["x0"] + d["x1"]) / 2)
coal_lbl = alt.Chart(coal).mark_text(
    baseline="top", dy=4, fontSize=10, color=INK).encode(
    x="mid:Q", y=alt.value(0), text=alt.value("Coalition"))

# grey wash over the projection years (2025+)
proj = pd.DataFrame({"x0": [2025], "x1": [data["year"].max() + 1]})
proj_bg = alt.Chart(proj).mark_rect(opacity=0.10, color="#676A86").encode(x="x0:Q", x2="x1:Q")

# line: solid actual, dashed projection (overlap at 2025 so they join)
actual = data[data["year"] <= 2025]
projline = data[data["year"] >= 2025]
line_a = alt.Chart(actual).mark_line(color=INK, strokeWidth=2.2).encode(
    x="year:Q", y=alt.Y("debt:Q", title="Gross debt, % of GDP"))
line_p = alt.Chart(projline).mark_line(color=INK, strokeWidth=2.2, strokeDash=[3, 3]).encode(
    x="year:Q", y="debt:Q")

chart = (bg + stripe_bg + proj_bg + line_a + line_p + coal_lbl).properties(
    width=640, height=340,
    title=alt.Title("Debt and the party in power",
                    subtitle="UK gross debt, % of GDP · shading shows government · dashed = IMF projection")
).resolve_scale(color="independent")

styles.save(chart, name="debt_by_party", svg=True)
chartt, name="debt_by_party", svg=True)
chart

alt.LayerChart(...)